# Lab2: 文本 + 框提示的交互式分割
## Interactive Segmentation with Text + Box Prompts

---

### 实验背景

在 Lab1 中我们试了纯文本提示，发现有些特殊建筑构件（燕尾脊、剪瓷雕、斗拱等）
仅靠自然语言描述难以准确定位。

本实验使用 **文本提示 + 框提示的组合方案**：

1. 在图片上用**滑块**调整框的位置和大小
2. 输入**文本描述**目标的语义特征
3. SAM 3 综合两种信息进行精确分割

### 框提示的格式

SAM 3 的 `add_geometric_prompt()` 接受归一化的 `[cx, cy, w, h]` 格式：

- `cx` = 框中心 x / 图片宽度（0~1）
- `cy` = 框中心 y / 图片高度（0~1）
- `w` = 框宽度 / 图片宽度（0~1）
- `h` = 框高度 / 图片高度（0~1）
- `label=True` 表示正例（前景），`label=False` 表示负例（背景）

In [2]:
# ========== 环境准备 ==========
import os
import sys
import csv
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from datetime import datetime

from IPython.display import display, clear_output
import ipywidgets as widgets

# ===== 修复 matplotlib 中文显示 =====
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'SimSun', 'FangSong', 'Arial Unicode MS', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示为方块
print(f"当前字体: {plt.rcParams['font.sans-serif'][0]}")
# ====================================

%matplotlib inline
print("环境就绪 ✅")

当前字体: Microsoft YaHei
环境就绪 ✅


In [4]:
# ========== 路径配置 ==========
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
CKPT_PATH = os.path.join(REPO_ROOT, "sam3.pt")
INPUT_DIR = os.path.join(REPO_ROOT, "Inputs", "RawImages")
CSV_PATH = os.path.join(INPUT_DIR, "试标注图像清单.csv")

LAB2_OUTPUT = os.path.join(REPO_ROOT, "Outputs", "Lab2_output")
os.makedirs(LAB2_OUTPUT, exist_ok=True)

if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f"权重未找到: {CKPT_PATH}")

# 读取 CSV
image_notes = {}
if os.path.exists(CSV_PATH):
    with open(CSV_PATH, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            image_notes[row["编号"]] = row

# 获取图片列表
image_files = sorted([
    f for f in os.listdir(INPUT_DIR) if f.endswith("_RawImage.png")
])
print(f"输入目录: {INPUT_DIR}")
print(f"图片数量: {len(image_files)}")
print(f"输出目录: {LAB2_OUTPUT}")
print(f"权重文件: {CKPT_PATH}")

输入目录: c:\Users\Legion\Desktop\WIE\SAM3_Labs\Inputs\RawImages
图片数量: 16
输出目录: c:\Users\Legion\Desktop\WIE\SAM3_Labs\Outputs\Lab2_output
权重文件: c:\Users\Legion\Desktop\WIE\SAM3_Labs\sam3.pt


In [5]:
# ========== 加载 SAM 3 模型 ==========
print("[⏳] 正在加载 SAM 3 模型...")
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"    设备: {device}")

model = build_sam3_image_model(
    checkpoint_path=CKPT_PATH,
    load_from_HF=False,
    device=device,
    eval_mode=True,
)
processor = Sam3Processor(model, device=device)
print("[✅] SAM 3 模型加载完成")
print(f"    参数量: {sum(p.numel() for p in model.parameters())/1e6:.0f}M")

[⏳] 正在加载 SAM 3 模型...
    设备: cuda
[✅] SAM 3 模型加载完成
    参数量: 841M


### 操作说明

运行下方单元格后，会出现一组控件：

1. **选择图片** → 下拉菜单
2. **输入文本提示** → 文本框（如 `"swallowtail ridge"`）
3. **调整框的位置和大小** → 四个滑块（x1, y1, x2, y2 像素坐标）
4. **点击"运行分割"按钮** → 执行 SAM 3 推理
5. **结果立刻显示** → 图片上方叠加 mask

> 💡 **技巧**：先看 CSV 描述知道目标在哪，再调滑块框住它

In [6]:
# ========== 交互式框选 + SAM 3 推理 ==========

# 当前加载的图片状态
_current_img = None
_current_img_id = None
_current_img_w = 800
_current_img_h = 600
_result_axes = None
_result_fig = None


def update_image_display(img_id):
    """切换图片并更新显示"""
    global _current_img, _current_img_id, _current_img_w, _current_img_h
    
    filename = f"{img_id}_RawImage.png"
    img_path = os.path.join(INPUT_DIR, filename)
    
    if os.path.exists(img_path):
        _current_img = Image.open(img_path).convert("RGB")
        _current_img_id = img_id
        _current_img_w, _current_img_h = _current_img.size
        
        # 更新滑块的 max 值
        slider_x2.max = _current_img_w
        slider_y2.max = _current_img_h
        slider_x1.max = _current_img_w - 10
        slider_y1.max = _current_img_h - 10
        
        # 显示描述
        notes = image_notes.get(img_id, {})
        desc_output.clear_output()
        with desc_output:
            print(f"📷 {filename} ({_current_img_w}×{_current_img_h})")
            print(f"🏗️  {notes.get('主要构件', '')}")
            print(f"📋 {notes.get('场景特点', '')}")
            print(f"⚠️  {notes.get('干扰项', '')}")
        
        # 刷新预览图
        update_preview(None)
    

def update_preview(change):
    """实时更新预览图上的框"""
    global _result_fig
    
    if _current_img is None:
        return
    
    x1 = slider_x1.value
    y1 = slider_y1.value
    x2 = slider_x2.value
    y2 = slider_y2.value
    
    preview_output.clear_output(wait=True)
    with preview_output:
        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
        ax.imshow(np.array(_current_img))
        
        # 画框
        rect = plt.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2, edgecolor='lime', facecolor='lime', alpha=0.2
        )
        ax.add_patch(rect)
        
        # 显示归一化坐标（用英文避免字体问题）
        cx = ((x1 + x2) / 2) / _current_img_w
        cy = ((y1 + y2) / 2) / _current_img_h
        bw = (x2 - x1) / _current_img_w
        bh = (y2 - y1) / _current_img_h
        ax.set_title(f"Normalized: cx={cx:.3f}, cy={cy:.3f}, w={bw:.3f}, h={bh:.3f}", fontsize=10)
        ax.axis("off")
        plt.tight_layout()
        plt.show()


def run_segmentation(btn):
    """执行文本+框分割"""
    if _current_img is None:
        print("[⚠️] 请先选择一张图片")
        return
    
    text_prompt = text_input.value.strip()
    x1 = slider_x1.value
    y1 = slider_y1.value
    x2 = slider_x2.value
    y2 = slider_y2.value
    
    # 归一化
    cx = ((x1 + x2) / 2) / _current_img_w
    cy = ((y1 + y2) / 2) / _current_img_h
    bw = (x2 - x1) / _current_img_w
    bh = (y2 - y1) / _current_img_h
    
    result_output.clear_output(wait=True)
    with result_output:
        print(f"[🎯] Box: ({x1},{y1})->({x2},{y2})")
        print(f"[💬] Text: {text_prompt}")
        print(f"[⚙️] Normalized: cx={cx:.3f}, cy={cy:.3f}, w={bw:.3f}, h={bh:.3f}")
        
        try:
            state = processor.set_image(_current_img)
            state = processor.set_text_prompt(prompt=text_prompt, state=state)
            state = processor.add_geometric_prompt(
                box=[cx, cy, bw, bh], label=True, state=state
            )
            
            masks = state["masks"]
            scores = state["scores"]
            
            print(f"[✅] Found {len(masks)} instances\n")
            
            for i in range(len(masks)):
                print(f"    #{i}: score={scores[i].item():.3f}, "
                      f"area={masks[i].sum().item() / (_current_img_w*_current_img_h)*100:.1f}%")
            
            # ---- 可视化 ----
            fig, axes = plt.subplots(1, 2, figsize=(14, 6))
            
            # 左：原图+框
            axes[0].imshow(np.array(_current_img))
            rect = plt.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor='lime', facecolor='none', linestyle='--'
            )
            axes[0].add_patch(rect)
            axes[0].set_title("Original + Box")
            axes[0].axis("off")
            
            # 右：mask 叠加
            overlay = np.zeros((_current_img_h, _current_img_w, 4), dtype=np.float32)
            cmap = plt.cm.tab10
            for i in range(len(masks)):
                m = masks[i].squeeze().cpu().numpy()
                color = cmap(i % 10)
                overlay[m] = color[:3] + (0.5,)
                
                # 标记分数
                ys, xs = np.where(m)
                if len(xs) > 0:
                    cy_m, cx_m = ys[len(ys)//2], xs[len(xs)//2]
                    axes[1].text(cx_m, cy_m, f"{scores[i].item():.2f}",
                        color='white', fontsize=9, ha='center', va='center',
                        bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.6))
            
            axes[1].imshow(np.array(_current_img))
            axes[1].imshow(overlay)
            axes[1].set_title(f"'{text_prompt}' + Box | {len(masks)} instances")
            axes[1].axis("off")
            
            plt.tight_layout()
            plt.show()
            
            # ---- 保存 ----
            timestamp = datetime.now().strftime("%H%M%S")
            save_dir = os.path.join(LAB2_OUTPUT, f"{_current_img_id}_{timestamp}")
            os.makedirs(save_dir, exist_ok=True)
            fig.savefig(os.path.join(save_dir, "result.png"), dpi=150, bbox_inches="tight")
            
            for i in range(len(masks)):
                mask_np = masks[i].squeeze().cpu().numpy().astype(np.uint8) * 255
                Image.fromarray(mask_np).save(
                    os.path.join(save_dir, f"mask_{i:02d}_{scores[i].item():.3f}.png")
                )
            print(f"\n[💾] Saved: {save_dir}")
            
        except Exception as e:
            import traceback
            traceback.print_exc()
            print(f"[❌] Inference failed: {e}")


# ========== 创建 UI ==========

# 图片选择
img_selector = widgets.Dropdown(
    options=[(f"{f.replace('_RawImage.png','')}", f.replace('_RawImage.png',''))
             for f in image_files],
    description='Select image:',
    layout={'width': '200px'}
)

# 文本提示
text_input = widgets.Text(
    value='swallowtail ridge',
    description='Text prompt:',
    layout={'width': '400px'}
)

# 框坐标滑块（像素坐标）
slider_x1 = widgets.IntSlider(value=100, min=0, max=700, step=1,
                              description='x1 (left):', layout={'width': '500px'})
slider_y1 = widgets.IntSlider(value=100, min=0, max=500, step=1,
                              description='y1 (top):', layout={'width': '500px'})
slider_x2 = widgets.IntSlider(value=300, min=10, max=800, step=1,
                              description='x2 (right):', layout={'width': '500px'})
slider_y2 = widgets.IntSlider(value=300, min=10, max=600, step=1,
                              description='y2 (bottom):', layout={'width': '500px'})

# 运行按钮
run_btn = widgets.Button(
    description='🏃 Run Segmentation',
    button_style='success',
    layout={'width': '220px', 'height': '40px'}
)
run_btn.on_click(run_segmentation)

# 输出区域
desc_output = widgets.Output()
preview_output = widgets.Output()
result_output = widgets.Output()

# ---- 布局 ----
box_grid = widgets.VBox([
    widgets.HBox([slider_x1, slider_x2]),
    widgets.HBox([slider_y1, slider_y2]),
])

controls = widgets.VBox([
    widgets.HBox([img_selector, text_input]),
    widgets.HTML("<b>Box coordinates (pixels):</b>"),
    box_grid,
    run_btn,
    desc_output,
])

# ---- 绑定事件 ----
img_selector.observe(
    lambda c: update_image_display(c.new) if c.new else None,
    names='value'
)
slider_x1.observe(update_preview, names='value')
slider_y1.observe(update_preview, names='value')
slider_x2.observe(update_preview, names='value')
slider_y2.observe(update_preview, names='value')

# ---- 显示 ----
display(controls)
display(widgets.HTML("<b>Box Preview:</b>"))
display(preview_output)
display(widgets.HTML("<b>Segmentation Result:</b>"))
display(result_output)

# 加载第一张图
if image_files:
    first_id = image_files[0].replace('_RawImage.png', '')
    img_selector.value = first_id

HTML(value='<b>Box Preview:</b>')

Output()

HTML(value='<b>Segmentation Result:</b>')

Output()

### Recommended Prompts

| Feature | Suggested Text Prompt | Box Placement |
|---------|---------------------|---------------|
| Swallowtail ridge | `swallowtail ridge`, `curved roof ridge` | Both ends of the roof |
| Horse-back gable | `horse-back gable` | Side wall |
| Porcelain inlay | `porcelain inlay`, `roof sculpture` | Roof decoration |
| Waterwheel frieze | `waterwheel frieze`, `colorful frieze` | Under-eaves painting |
| Dougong bracket | `wooden bracket`, `dougong bracket` | Bracket location |
| Stone door frame | `stone door frame` | Door frame |

### Notes

- Adjust the box sliders to include the target region, then click **Run Segmentation**
- Results are auto-saved to `Outputs/Lab2_output/<image_id>_<timestamp>/`
- Try different text prompts with the same box to compare results